# ATiG 2026: LD score regression
29 October 2026 · About 75 minutes · Work in pairs

We will read three published GWAS, match them to one LD reference, and use LD score
regression (LDSC) to estimate SNP heritability and genetic correlations.

Run the cells in order. Change the settings marked `# TRY` and discuss the questions.
The setup downloads about 240 MB of GWAS results. [Open in Colab](https://colab.research.google.com/github/bvilhjal/ATIG_2026/blob/main/teaching_days/2026-10-29/LDSC/LDSC_practical.ipynb)
and choose File → Save a copy in Drive. Use a Python 3 runtime.

## Setup
Run the three cells. No editing needed. The first installs [PLDSC](https://github.com/bvilhjal/PLDSC),
which brings [allelign](https://github.com/bvilhjal/allelign) and the LD scores.
The download takes a minute or two.

In [ ]:
%pip install -q "pldsc @ git+https://github.com/bvilhjal/PLDSC@v0.1.1"

In [ ]:
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import allelign
from pldsc import load_ld_reference, ldsc_chi2_keep, ldsc_h2_sumstats, ldsc_rg_sumstats

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
#@title Download the GWAS files (no editing needed)
SOURCES = {  # trait: (file name, URL, SHA-256 of the file used here)
    'BMI': ('bmi_giant_ukb_2018.txt.gz',
            'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/'
            'GCST006001-GCST007000/GCST006900/Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz',
            '0d6ed0ea97870916b830ccae349df94ca6f3cc68c025c79e784729af7f7136a4'),
    'TG': ('tg_glgc_2013.txt.gz',
           'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_TG.txt.gz',
           '37087a0015c3211f5696616ad276eb4b5fae7219e0b8a2f37a5ed86b626c5f0b'),
    'DBP': ('dbp_evangelou_2018.txt.gz',
            'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/'
            'GCST006001-GCST007000/GCST006630/Evangelou_30224653_DBP.txt.gz',
            'c55cfc8b50abee8e13ef88d70ab52678b1f45b45cf5974bdab91f8c8b361fd59'),
}
folder = Path('gwas_downloads')
folder.mkdir(exist_ok=True)
paths = {}
for trait, (name, url, sha256) in SOURCES.items():
    path = folder / name
    if not path.exists():
        urllib.request.urlretrieve(url, folder / 'partial')
        (folder / 'partial').rename(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha256:
        raise ValueError(f'{path} is not the file used here; delete it and rerun')
    paths[trait] = path
    print(f'{trait}: {name}, {path.stat().st_size / 1e6:.0f} MB')

## Q1. Look at the files

Each consortium uses its own format. Tables 1–3 show the first rows of each file.

- BMI: [Yengo et al. (2018)](https://pubmed.ncbi.nlm.nih.gov/30124842/), GIANT and UK Biobank, about 700,000 people.
- TG (triglycerides): [Willer et al. (2013)](https://pubmed.ncbi.nlm.nih.gov/24097068/), Global Lipids Genetics Consortium, about 90,000 people for most SNPs.
- DBP (diastolic blood pressure): [Evangelou et al. (2018)](https://pubmed.ncbi.nlm.nih.gov/30224653/), UK Biobank and ICBP, about 750,000 people.

For each file, find the effect allele, the other allele, the effect estimate, its standard
error, the sample size and the genomic position. Which file has no rsIDs? Which has no
position column? What else differs?

In [ ]:
for number, (trait, path) in enumerate(paths.items(), start=1):
    print(f'Table {number}. The first rows of the {trait} file, as text.')
    display(pd.read_csv(path, sep=r'\s+', nrows=3, dtype=str))

## Q2. Read the files

allelign recognises common column names and reads each file into the same layout.
Table 4 shows which columns it used. The DBP file has no rsIDs: its `MarkerName` is
chromosome:position:type on GRCh37, so we split it to get positions. DBP is then
matched by position.

Which fields were not found? For DBP, allelign used `N_effective` (sample size times
imputation quality) rather than `TotalSampleSize`. Why does the choice of N matter
for heritability?

In [ ]:
mappings = {trait: allelign.inspect_columns(path)['mapping'] for trait, path in paths.items()}
print('Table 4. File columns used by allelign; blank if not found.')
display(pd.DataFrame(mappings).fillna(''))

In [ ]:
gwas = {trait: allelign.read_sumstats(path) for trait, path in paths.items()}  # About 30 s.

# Chromosome and position from DBP's MarkerName, e.g. 10:100000625:SNP.
dbp = gwas['DBP']
marker = pd.Series(dbp.id).str.split(':', expand=True)
gwas['DBP'] = allelign.Sumstats(
    id=dbp.id, chrom=marker[0].to_numpy(), pos=marker[1].astype(int).to_numpy(),
    ea=dbp.ea, oa=dbp.oa, beta=dbp.beta, se=dbp.se, n_eff=dbp.n_eff, eaf=dbp.eaf, info=dbp.info)

for trait, sumstats in gwas.items():
    print(f'{trait}: {len(sumstats):,} variants, median N {np.median(sumstats.n_eff):,.0f}')

## Q3. Match to the LD reference

LDSC needs an LD score for every SNP. We use scores for about 1.2 million HapMap3 SNPs,
calculated from 489 Europeans in the 1000 Genomes Project; they come with PLDSC. PLDSC
leaves out about 30,000 of them that lie in regions with unusually long-range LD, such
as the MHC.
allelign first removes duplicates and variants with low sample size. It then aligns each
GWAS to the reference allele `a1`, changing the sign when a file's effect allele is `a2`
(Table 5).

Why is only a small part of the DBP file used? Table 6 follows rs1421085 in *FTO*
through the three files. Why must all traits refer to the same allele before we
correlate them?

In [ ]:
ref = load_ld_reference('hm3-eur')
print(ref)

aligned = {}
counts = {}
for trait, sumstats in gwas.items():
    keep, qc = allelign.qc_sumstats(sumstats)
    aligned[trait] = allelign.harmonize(sumstats.subset(keep), ref.variants)
    log = aligned[trait].log
    counts[trait] = {'in file': len(sumstats), 'after QC': qc['n_kept'],
                     'matched': log['n_matched'], 'sign changed': log['n_flipped']}
print('Table 5. Variants kept at each step.')
display(pd.DataFrame(counts))

In [ ]:
snp = 'rs1421085'  # TRY: rs1260326 (GCKR) or rs13107325 (SLC39A8)
row = np.flatnonzero(ref.variants.id == snp)[0]
chrom, pos = ref.variants.chrom[row], ref.variants.pos[row]
print(f'{snp}: chromosome {chrom}, position {pos:,}; '
      f'reference a1 = {ref.variants.a1[row]}, a2 = {ref.variants.a2[row]}')

rows = {}
for trait, sumstats in gwas.items():
    # rsID for BMI and TG, position for DBP.
    i = np.flatnonzero((sumstats.id == snp) | ((sumstats.chrom == chrom) & (sumstats.pos == pos)))[0]
    j = np.flatnonzero(aligned[trait].var_index == row)[0]
    rows[trait] = {'effect allele': sumstats.ea[i], 'other allele': sumstats.oa[i],
                   'z in file': sumstats.beta[i] / sumstats.se[i],
                   'z for a1': aligned[trait].beta[j] / aligned[trait].se[j]}
snp_table = pd.DataFrame(rows).T.astype({'z in file': float, 'z for a1': float})
print(f'Table 6. {snp} in the three GWAS.')
display(snp_table)

## Q4. Estimate SNP heritability

LDSC models the expected χ² statistic of SNP $j$ as

$$E[\chi^2_j] = 1 + Na + \frac{N h^2}{M}\,\ell_j, \qquad (1)$$

where $\ell_j$ is its LD score, $N$ the sample size, $M$ the number of SNPs the
heritability $h^2$ refers to, and $a$ the inflation from confounding, such as population
stratification. Figure 1 shows the mean χ² against LD score; Table 7 has the fits.

Which trait has the highest heritability? Compare each intercept with λGC, the median χ²
divided by 0.455 (the median expected without any association). Which tells you more
about confounding? Why might TG's intercept be below one? Then set `M = ref.m_snps` and
rerun both cells. What changes, and what does not?

In [ ]:
M = ref.m_snps_5_50  # SNPs with MAF 5–50%, as in the original LDSC. TRY: ref.m_snps

fits = {trait: ldsc_h2_sumstats(sumstats, ref.variants, ref.scores, m_snps=M).estimate
        for trait, sumstats in gwas.items()}  # About 30 s.

table = {}
for trait, fit in fits.items():
    z = aligned[trait].beta / aligned[trait].se
    table[trait] = {'h2': fit.h2, 'SE': fit.h2_se, 'intercept': fit.intercept,
                    'intercept SE': fit.intercept_se, 'mean chi2': fit.mean_chisq,
                    'lambda GC': allelign.genomic_inflation(z=z)}
print(f'Table 7. SNP heritability with M = {M:,}.')
display(pd.DataFrame(table).T)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, (trait, fit) in zip(axes, fits.items()):
    h = aligned[trait]
    chi2 = (h.beta / h.se)**2
    used = ldsc_chi2_keep(chi2, h.n_eff)[0]  # The fit drops SNPs with very large chi2.
    snps = pd.DataFrame({'ld_score': ref.scores[h.var_index][used], 'chi2': chi2[used]})
    snps = snps[snps['ld_score'] < snps['ld_score'].quantile(0.99)]  # Readable x-axis.
    snps['bin'] = pd.qcut(snps['ld_score'], 50, labels=False)  # 50 bins of equal size.
    means = snps.groupby('bin').mean()
    ax.scatter(means['ld_score'], means['chi2'], s=10)

    # Equation (1) with the fitted intercept and h2, at the median sample size.
    grid = np.linspace(0, means['ld_score'].max(), 100)
    ax.plot(grid, fit.intercept + np.median(h.n_eff) * fit.h2 / M * grid, color='darkorange')
    ax.set_title(trait)
    ax.set_xlabel('LD score')
axes[0].set_ylabel('Mean χ²')
fig.suptitle('Figure 1. Mean χ² in LD-score bins, with the LDSC fit (orange)')
plt.tight_layout()
plt.show()

## Q5. Genetic correlation between BMI and TG

Bivariate LDSC uses the product of the two traits' z scores:

$$E[z_{1j} z_{2j}] = \frac{\sqrt{N_1 N_2}\,\rho_g}{M}\,\ell_j + \frac{N_s\,\rho}{\sqrt{N_1 N_2}}, \qquad (2)$$

where $\rho_g$ is the genetic covariance, $N_s$ the number of people in both GWAS and
$\rho$ the phenotypic correlation among them. The genetic correlation is
$r_g = \rho_g / \sqrt{h^2_1 h^2_2}$.

Is the BMI–TG genetic correlation clearly different from zero? What does the cross-trait
intercept tell you? The plain correlation of z scores across SNPs is much smaller than
$r_g$. Why?

In [ ]:
pair = ('BMI', 'TG')
fit = ldsc_rg_sumstats(gwas[pair[0]], gwas[pair[1]], ref.variants, ref.scores,
                       m_snps=M, ncores=2).estimate
print(f'{pair[0]}–{pair[1]} genetic correlation: {fit.rg:.3f} (SE {fit.rg_se:.3f})')
print(f'Genetic covariance: {fit.gcov:.4f}')
print(f'Cross-trait intercept: {fit.gcov_intercept:.3f} (SE {fit.gcov_intercept_se:.3f})')

# For comparison: the plain correlation of z scores, ignoring LD.
shared, i, j = np.intersect1d(aligned[pair[0]].var_index, aligned[pair[1]].var_index,
                              return_indices=True)
z1 = aligned[pair[0]].beta[i] / aligned[pair[0]].se[i]
z2 = aligned[pair[1]].beta[j] / aligned[pair[1]].se[j]
print(f'Correlation of z scores over {len(shared):,} SNPs: {np.corrcoef(z1, z2)[0, 1]:.3f}')

## Q6. Add blood pressure

Table 8 shows all three pairs. UK Biobank is part of both the BMI and the DBP GWAS.
Evangelou et al. analysed blood pressure adjusted for age, sex and BMI.

Is the BMI–DBP result what you expected? How can adjusting DBP for BMI change its genetic
correlation with BMI? Why is the BMI–DBP cross-trait intercept close to zero despite the
overlap?

In [ ]:
pairs = [('BMI', 'TG'), ('BMI', 'DBP'), ('TG', 'DBP')]
table = {}
for trait1, trait2 in pairs:  # About a minute.
    estimate = ldsc_rg_sumstats(gwas[trait1], gwas[trait2], ref.variants, ref.scores,
                                m_snps=M, ncores=2).estimate
    table[f'{trait1}–{trait2}'] = {'rg': estimate.rg, 'SE': estimate.rg_se,
                                   'cross-trait intercept': estimate.gcov_intercept,
                                   'intercept SE': estimate.gcov_intercept_se}
print('Table 8. Genetic correlations.')
display(pd.DataFrame(table).T)

## Q7. Conclude

Write three sentences: report the BMI–TG genetic correlation with its standard error,
explain why it does not show that higher BMI raises triglycerides, and say what the
BMI–DBP result teaches you about using published GWAS results.

## Going further (optional)

**More traits.** The cell below adds one more GWAS in a format you have already read:
other lipids from the same consortium as TG, or height from the same study as BMI.
It fits its heritability and its genetic correlations with BMI, TG and DBP. Which
pairs would you expect to be correlated? Look for other European-ancestry GWAS with
rsIDs or GRCh37 positions, for example in the [GWAS Catalog](https://www.ebi.ac.uk/gwas/downloads/summary-statistics),
and add them in the same way. Check `allelign.inspect_columns` first.

**Ask an AI model to implement LDSC.** Give an AI assistant (Claude, ChatGPT or another)
the aligned data for one trait and ask for univariate LDSC from scratch in NumPy,
with LDSC's regression weights and a block-jackknife standard error:

```python
h = aligned['BMI']
z, n, ell = h.beta / h.se, h.n_eff, ref.scores[h.var_index]
```

Compare its $h^2$ and intercept with Table 7. Then read the code: which M does it use,
how are the weights defined, and what happens to SNPs with very large χ²? Can you make
it agree with PLDSC? Then try bivariate LDSC.

In [ ]:
MORE = {  # Same formats as TG and BMI.
    'HDL': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_HDL.txt.gz',
    'LDL': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_LDL.txt.gz',
    'TC': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_TC.txt.gz',
    'Height': 'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/'
              'GCST006001-GCST007000/GCST006901/Meta-analysis_Wood_et_al+UKBiobank_2018.txt.gz',
}
trait = 'HDL'  # TRY: 'LDL', 'TC' or 'Height'
path = folder / MORE[trait].split('/')[-1]
if not path.exists():
    urllib.request.urlretrieve(MORE[trait], folder / 'partial')
    (folder / 'partial').rename(path)
gwas[trait] = allelign.read_sumstats(path)
print(trait, ldsc_h2_sumstats(gwas[trait], ref.variants, ref.scores, m_snps=M).estimate)
for other in ('BMI', 'TG', 'DBP'):
    fit = ldsc_rg_sumstats(gwas[other], gwas[trait], ref.variants, ref.scores,
                           m_snps=M, ncores=2).estimate
    print(f'{other}–{trait}: rg {fit.rg:.3f} (SE {fit.rg_se:.3f})')

## Data and method notes

The GWAS files are downloaded and checked against the SHA-256 checksums in the setup cell.
All three are meta-analyses of European-ancestry samples with GRCh37 positions. BMI (GIANT)
and DBP (discovery stage) come from the GWAS Catalog
([GCST006900](https://www.ebi.ac.uk/gwas/studies/GCST006900),
[GCST006630](https://www.ebi.ac.uk/gwas/studies/GCST006630)), TG from the
[Global Lipids Genetics Consortium](https://csg.sph.umich.edu/willer/public/lipids2013/)
(joint GWAS and Metabochip analysis). The TG studies were corrected for genomic control,
and the meta-analysis P values were corrected again. For DBP, the UK Biobank statistics
were divided by their LDSC intercept (1.219) and the ICBP studies corrected for genomic
control before the meta-analysis. Further details are in [`data/sources.json`](data/sources.json).

The LD scores are the 1000 Genomes Phase 3 European scores for HapMap3 SNPs released with
LDSC ([Zenodo 7768714](https://doi.org/10.5281/zenodo.7768714)), packaged in PLDSC as
`hm3-eur`. The set has almost no strand-ambiguous A/T or C/G SNPs (4 of 1.19 million),
and allelign drops those. PLDSC fits equations (1) and (2) in one step, after dropping SNPs
with χ² above max(80, N/1000), and uses these LD scores for the regression weights; standard
errors come from a block jackknife. PLDSC leaves out allelign's GRCh37 long-range LD
regions by default (`load_ld_reference('hm3-eur', include_long_range_ld=True)` keeps them).
The original LDSC software instead uses a two-step estimator, and its regression weights
leave out only the MHC. Its estimates differ
somewhat from these, most for TG, whose h² depends on how SNPs with very large χ² are
handled. The plain z-score correlation in Q5 is shown only for contrast.

Background: [Bulik-Sullivan et al. (2015a)](https://pubmed.ncbi.nlm.nih.gov/25642630/) on LDSC
and [Bulik-Sullivan et al. (2015b)](https://pubmed.ncbi.nlm.nih.gov/26414676/) on genetic
correlation.